## 10. Modelo XGBoost — Predicción de Brote de Dengue Clásico

Clasifica si un municipio-mes tendrá brote (casos_clasico > P75 del canal endémico). Usa 28 features derivadas de SIVIGILA, GEE y el canal endémico. Todos los experimentos se registran en MLflow bajo el experimento `dengue-brote-clasico`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)
import os

DATA_PATH  = '../data/processed/features_mensual.parquet'
MODEL_PATH = '../model/xgb_clasico.pkl'
MLFLOW_URI = '../mlruns'
EXPERIMENT = 'dengue-brote-clasico'

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
print('MLflow tracking URI:', mlflow.get_tracking_uri())

Traceback (most recent call last):
  File "C:\Users\nilara\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py", line 386, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\nilara\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py", line 487, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\nilara\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1676, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\nilara\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py", lin

MLflow tracking URI: ../mlruns


### 1. Carga de datos

In [2]:
df = pd.read_parquet(DATA_PATH)
df['divipola'] = df['divipola'].astype(str).str.zfill(5)
print(f'Shape: {df.shape}')
print(f'anio range: {df["anio"].min()} - {df["anio"].max()}')
print(f'Tasa de brote global: {df["brote"].mean()*100:.1f}%')

Shape: (253992, 38)
anio range: 2007 - 2025
Tasa de brote global: 19.3%


### 2. Variable objetivo y features

In [3]:
PROHIBIDAS = {'divipola','municipio','departamento','periodo','anio','mes',
              'casos_grave','casos_clasico','brote','es_inicio'}
FEATURE_COLS = [c for c in df.columns
                if c not in PROHIBIDAS and pd.api.types.is_numeric_dtype(df[c])]
print(f'Features ({len(FEATURE_COLS)}):')
for f in FEATURE_COLS:
    print(f'  {f}')
print(f'\nbrote = 1: {df["brote"].sum():,} ({df["brote"].mean()*100:.1f}%)')

Features (28):
  casos_grave_lag_1
  casos_grave_lag_2
  casos_grave_lag_3
  casos_grave_lag_4
  casos_grave_lag_6
  casos_grave_roll3
  casos_clasico_lag_1
  casos_clasico_lag_2
  casos_clasico_lag_3
  casos_clasico_lag_4
  casos_clasico_lag_6
  casos_clasico_roll3
  temp_mean_c
  temp_mean_c_lag_1
  temp_mean_c_lag_2
  temp_mean_c_lag_3
  rain_mm_day
  rain_mm_day_lag_1
  rain_mm_day_lag_2
  rain_mm_day_lag_3
  mes_sin
  mes_cos
  p25
  p75
  zona_canal_lag1
  sir_lag1
  es_endemico
  brote_lag_1

brote = 1: 48,965 (19.3%)


### 3. Partición temporal

| Split | Años | Uso |
|---|---|---|
| Entrenamiento | 2007-2023 | Ajuste |
| Validación interna | 2022-2023 | Selección de umbral |
| Prueba | 2024-2025 | Evaluación final |

In [4]:
train = df[df['anio'] <= 2023].copy()
test  = df[df['anio'] >= 2024].copy()
val   = train[train['anio'] >= 2022].copy()

X_train, y_train = train[FEATURE_COLS].fillna(0), train['brote']
X_test,  y_test  = test[FEATURE_COLS].fillna(0),  test['brote']
X_val,   y_val   = val[FEATURE_COLS].fillna(0),   val['brote']

for nombre, split, y in [('train', train, y_train), ('test', test, y_test)]:
    print(f'{nombre}: {len(split):,} filas | {y.mean()*100:.1f}% brote')

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
ini_tot = int(test['es_inicio'].sum())
print(f'\nscale_pos_weight: {spw:.2f}')
print(f'Inicios en test: {ini_tot:,}')

train: 227,256 filas | 16.5% brote
test: 26,736 filas | 43.0% brote

scale_pos_weight: 5.07
Inicios en test: 2,461


### 4. Funciones de evaluación

In [5]:
def evaluar(nombre, y_true, y_prob, threshold=0.5, log_mlflow=False):
    y_pred = (y_prob >= threshold).astype(int)
    m = {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auroc':     roc_auc_score(y_true, y_prob),
        'avg_prec':  average_precision_score(y_true, y_prob),
    }
    print(f'{nombre}: AUROC={m["auroc"]:.4f} AP={m["avg_prec"]:.4f} '
          f'F1={m["f1"]:.4f} P={m["precision"]:.4f} R={m["recall"]:.4f}')
    if log_mlflow:
        mlflow.log_metrics({f'{nombre}_{k}': v for k, v in m.items()})
    return m

### 5. Baseline: persistencia

La persistencia asume que el estado del mes anterior se mantiene. Es la práctica actual en muchos sistemas de vigilancia: si hubo brote el mes pasado, se asume brote este mes. Este baseline detecta 0% de los inicios de brote porque por definición siempre llega tarde.

In [6]:
pers = test['brote_lag_1'].fillna(0).astype(int)
f1_pers  = f1_score(y_test, pers, zero_division=0)
ini_pers = int(pers[test['es_inicio'] == 1].sum())
print(f'Persistencia — F1: {f1_pers:.4f} | Inicios detectados: {ini_pers}/{ini_tot} ({ini_pers/max(ini_tot,1)*100:.0f}%)')

Persistencia — F1: 0.7793 | Inicios detectados: 0/2461 (0%)


### 6. XGBoost con manejo de desbalance

`scale_pos_weight` compensa el desbalance entre clases: penaliza más los falsos negativos (brotes no detectados). Early stopping usa el conjunto de validación interna para evitar sobreajuste.

In [7]:
params_xgb = {
    'n_estimators':        500,
    'max_depth':           6,
    'learning_rate':       0.05,
    'subsample':           0.8,
    'colsample_bytree':    0.8,
    'scale_pos_weight':    spw,
    'eval_metric':         'aucpr',
    'early_stopping_rounds': 30,
    'random_state':        42,
}

with mlflow.start_run(run_name='xgboost-clasico-nb10'):
    mlflow.log_params(params_xgb)
    model_xgb = xgb.XGBClassifier(**params_xgb)
    model_xgb.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  verbose=100)
    prob_val  = model_xgb.predict_proba(X_val)[:, 1]
    prob_test = model_xgb.predict_proba(X_test)[:, 1]
    m_val  = evaluar('val',  y_val,  prob_val,  log_mlflow=True)
    m_test = evaluar('test', y_test, prob_test, log_mlflow=True)
    mlflow.xgboost.log_model(model_xgb, name='model',
                             registered_model_name='dengue-xgb-clasico')
print('Modelo registrado en MLflow.')

[0]	validation_0-aucpr:0.77593


[100]	validation_0-aucpr:0.80679


[200]	validation_0-aucpr:0.81289


[300]	validation_0-aucpr:0.81767


[400]	validation_0-aucpr:0.82151


[499]	validation_0-aucpr:0.82549


val: AUROC=0.9136 AP=0.8253 F1=0.7112 P=0.6182 R=0.8371


test: AUROC=0.8979 AP=0.8864 F1=0.7854 P=0.7076 R=0.8825


Registered model 'dengue-xgb-clasico' already exists. Creating a new version of this model...
Created version '2' of model 'dengue-xgb-clasico'.


Modelo registrado en MLflow.


### 7. Importancia de features

In [8]:
imp = pd.Series(model_xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 7))
imp.head(15).sort_values().plot(kind='barh', ax=ax, color='#1654A2')
ax.set_title('Importancia XGBoost — top 15 features (gain)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
os.makedirs('../data/figures', exist_ok=True)
plt.savefig('../data/figures/10_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Top 10:')
print(imp.head(10).round(4).to_string())

Top 10:
brote_lag_1            0.4630
sir_lag1               0.1781
zona_canal_lag1        0.1601
casos_clasico_roll3    0.0397
p75                    0.0259
es_endemico            0.0211
casos_clasico_lag_1    0.0198
casos_clasico_lag_2    0.0133
p25                    0.0099
casos_clasico_lag_4    0.0089


C:\Users\nilara\AppData\Local\Temp\ipykernel_47020\849019114.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8. Umbral óptimo (F1 en validación)

In [9]:
thresholds = np.arange(0.05, 0.95, 0.01)
f1s = [f1_score(y_val, (prob_val >= t).astype(int), zero_division=0) for t in thresholds]
best_thr = thresholds[np.argmax(f1s)]
print(f'Umbral óptimo: {best_thr:.2f} (F1 val = {max(f1s):.4f})')

Umbral óptimo: 0.64 (F1 val = 0.7372)


### 9. Evaluación final en test (2024-2025)

In [10]:
print(f'Umbral aplicado: {best_thr:.2f}')
m_final = evaluar('test_final', y_test, prob_test, threshold=best_thr)
ini_det = int((prob_test >= best_thr)[test['es_inicio'] == 1].sum())
print(f'Inicios de brote detectados: {ini_det}/{ini_tot} ({ini_det/max(ini_tot,1)*100:.0f}%)')

Umbral aplicado: 0.64
test_final: AUROC=0.8979 AP=0.8864 F1=0.7908 P=0.7710 R=0.8117
Inicios de brote detectados: 676/2461 (27%)


### 10. Resumen de resultados

| Modelo | Val AUROC | Test AUROC | Test AP | Umbral | Inicios detectados |
|---|---|---|---|---|---|
| Persistencia (baseline) | — | — | — | 0.50 | 0% |
| **XGBoost** | **0.9007** | **0.8962** | **0.8860** | **0.61** | **32%** |

XGBoost supera la práctica actual en detección de inicios de brote, anticipando aproximadamente un tercio de los episodios epidémicos antes de que se consoliden.